In [ ]:

!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install scipy

!pip install matplotlib
!pip install seaborn
!pip install plotly

!pip install nltk
!pip install spacy
!pip install keybert

!pip install torch torchvision torchaudio
!pip install transformers
!pip install sentence-transformers

!pip install faiss-cpu

!pip install pymupdf
!pip install pypdf
!pip install pdfplumber

!pip install streamlit

!pip install tqdm
!pip install ipywidgets
!pip install datasets
!pip install sentencepiece

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from tqdm import tqdm


df = pd.read_csv("arXiv_scientific_dataset.csv")
df=df.head(20000)

df.head()

df.shape
df.info()

df.isnull().sum()

df.duplicated().sum()

df = df.drop_duplicates()

df = df.dropna(subset=["title", "summary"])

df.reset_index(drop=True, inplace=True)

plt.figure(figsize=(10,5))

df["category"].value_counts().head(10).plot(kind="bar")

plt.title("Top 10 Research Categories")
plt.xlabel("Category")
plt.ylabel("Number of Papers")

plt.xticks

plt.show()

import re
import string

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["processed_text"] = (
    df["title"] + " " + df["summary"]
).apply(preprocess)

df[["title", "processed_text"]].head()

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["processed_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True  
)

embeddings.shape

np.save("embeddings.npy", embeddings)
embeddings = np.load("embeddings.npy")

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pca = PCA(n_components=2)

pca_embeddings = pca.fit_transform(embeddings)

pca_embeddings.shape

plt.figure(figsize=(8,6))

plt.scatter(
    pca_embeddings[:,0],
    pca_embeddings[:,1],
    s=8,
    alpha=0.6
)

plt.title("PCA Projection of Research Papers")

plt.xlabel("Principal Component 1")

plt.ylabel("Principal Component 2")

plt.show()

kmeans = KMeans(
    n_clusters=10,
    random_state=42
)

clusters = kmeans.fit_predict(embeddings)

df["Cluster"] = clusters

df[["title","category","Cluster"]].head()

plt.figure(figsize=(8,6))

plt.scatter(
    pca_embeddings[:,0],
    pca_embeddings[:,1],
    c=clusters,
    s=8
)

plt.title("K-Means Clusters")

plt.xlabel("Principal Component 1")

plt.ylabel("Principal Component 2")

plt.show()

df["Cluster"].value_counts().sort_index()

import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

def search_papers(query, top_k=5):

    query_embedding = model.encode([preprocess(query)])

    distances, indices = index.search(query_embedding, top_k)

    return df.iloc[indices[0]][["title", "category", "authors"]]

search_papers("Deep Learning for Medical Image Analysis")

search_papers("Cyber Security using Artificial Intelligence")

def recommend_papers(paper_index, top_k=5):

    distances, indices = index.search(
        embeddings[paper_index].reshape(1,-1),
        top_k + 1
    )

    recommendations = df.iloc[
        indices[0][1:]
    ][["title","category","authors"]]

    return recommendations

faiss.write_index(index, "faiss_index.index")

import spacy

nlp = spacy.load("en_core_web_sm")

sample_text = df.loc[7, "summary"]

doc = nlp(sample_text)

print(sample_text)

entities = []

for ent in doc.ents:
    entities.append([ent.text, ent.label_])

entity_df = pd.DataFrame(
    entities,
    columns=["Entity", "Type"]
)

entity_df

entity_df["Type"].value_counts()

def extract_entities(text):

    doc = nlp(text)

    return [
        (ent.text, ent.label_)
        for ent in doc.ents
    ]

df.loc[:99, "Entities"] = df.loc[:99, "summary"].apply(extract_entities)

df[["title", "Entities"]].head()

from keybert import KeyBERT

kw_model = KeyBERT(model)

sample_text = df.loc[7, "summary"]

keywords = kw_model.extract_keywords(
    sample_text,
    keyphrase_ngram_range=(1,2),
    stop_words="english",
    top_n=10
)

keywords

keyword_df = pd.DataFrame(
    keywords,
    columns=["Keyword","Score"]
)

keyword_df

def extract_keywords(text):
    return kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1,2),
        stop_words="english",
        top_n=5
    )

keyword_df = df.head(100).copy()

keyword_df["Keywords"] = keyword_df["summary"].apply(extract_keywords)

keyword_df.loc[7, ["title", "Keywords"]]

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df["category"])

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = SGDClassifier(
    loss="log_loss",
    max_iter=300,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
accuracy

print(classification_report(
    y_test,
    y_pred,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(cm, cmap="Blues")

plt.title("Confusion Matrix - Category Classification")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    framework="pt",
    device=0
)

sample_text = df.loc[7, "summary"]

summary = summarizer(
    sample_text,
    max_length=60,
    min_length=20,
    do_sample=False
)

summary[0]["summary_text"]

def summarize_texts_batch(text_list):
    summaries = summarizer(
        text_list,
        max_length=60,
        min_length=20,
        num_beams=1,
        do_sample=False,
        batch_size=4
    )
    return [s["summary_text"] for s in summaries]

summary_df = df.head(20).copy()

summary_df["Generated_Summary"] = summarize_texts_batch(summary_df["summary"].tolist())

summary_df[["title", "summary", "Generated_Summary"]]

print("Total Papers:", len(df))
print("Categories:", df["category"].nunique())
print("Clusters:", df["Cluster"].nunique())
print("Embedding Dimension:", embeddings.shape[1])
print("FAISS Index Size:", index.ntotal)
print("Classification Accuracy:", round(accuracy * 100, 2), "%")

!pip install -q langchain langchain-groq

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    api_key=groq_api_key
)


chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant that answers questions about scientific papers concisely."),
    ("human", "{question}")
])

parser = StrOutputParser()

chat_chain = chat_prompt | llm | parser


chat_chain.invoke({"question": "What is dependency-directed backtracking?"})


def retrieve_context(query, top_k=5):
    results = search_papers(query, top_k=top_k)
    context = "\n\n".join(
        f"Title: {row.title}\nCategory: {row.category}\nAuthors: {row.authors}"
        for row in results.itertuples()
    )
    return context, results


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user question using only the retrieved paper context below. Mention paper titles where relevant. If the context is insufficient, say so."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

rag_chain = rag_prompt | llm | parser


def rag_answer(query, top_k=5):
    context, results = retrieve_context(query, top_k=top_k)
    answer = rag_chain.invoke({"context": context, "question": query})
    return answer, results


answer, retrieved = rag_answer("What recent papers discuss reinforcement learning for robotics?")
print(answer)


retrieved


DOMAIN_PATTERNS = {
    "ALGORITHM": r"\b([A-Z][a-zA-Z]*(?:Net|GAN|BERT|GPT|CNN|RNN|LSTM|Transformer))\b",
    "METRIC": r"\b(accuracy|precision|recall|F1[- ]score|AUC|BLEU|perplexity)\b",
    "DATASET": r"\b([A-Z][a-zA-Z0-9]+(?:-[0-9]+)?)(?=\s(?:dataset|corpus|benchmark))"
}

def regex_entities(text):
    found = []
    for label, pattern in DOMAIN_PATTERNS.items():
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            found.append((m.group(0), label, 0.7))
    return found

def spacy_entities_scored(text):
    doc = nlp(text)
    return [(ent.text, ent.label_, 0.85) for ent in doc.ents]


ner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract named entities (methods, models, datasets, organizations, people) from the text. Return each on its own line in the format: entity | label. No extra commentary."),
    ("human", "{text}")
])

ner_chain = ner_prompt | llm | parser

def llm_entities(text):
    raw = ner_chain.invoke({"text": text})
    entities = []
    for line in raw.strip().split("\n"):
        if "|" in line:
            ent, label = line.split("|", 1)
            entities.append((ent.strip(), label.strip(), 0.6))
    return entities


def hybrid_ner(text):
    candidates = spacy_entities_scored(text) + regex_entities(text) + llm_entities(text)

    merged = {}
    for ent_text, label, score in candidates:
        key = ent_text.lower().strip()
        if key not in merged or score > merged[key][1]:
            merged[key] = (label, score)

    return pd.DataFrame(
        [(k, v[0], v[1]) for k, v in merged.items()],
        columns=["Entity", "Type", "Confidence"]
    ).sort_values("Confidence", ascending=False).reset_index(drop=True)


hybrid_ner(df.loc[7, "summary"])


def research_agent(query, top_k=5):
    plan = (
        f"Find papers relevant to '{query}', extract entities/keywords from the top result, "
        "summarize it, then answer the question using retrieved context."
    )

    context, retrieved = retrieve_context(query, top_k=top_k)

    top_title = retrieved.iloc[0]["title"]
    top_text = df.loc[df["title"] == top_title, "summary"].values[0]
    entities = hybrid_ner(top_text)
    keywords = extract_keywords(top_text)

    generated_summary = summarize_texts_batch([top_text])[0]

    final_answer = rag_chain.invoke({"context": context, "question": query})

    return {
        "plan": plan,
        "retrieved_papers": retrieved,
        "top_paper_summary": generated_summary,
        "entities": entities,
        "keywords": keywords,
        "answer": final_answer
    }


result = research_agent("How is deep learning used in medical image analysis?")

print("PLAN:", result["plan"])
print("\nFINAL ANSWER:\n", result["answer"])
print("\nTOP PAPER SUMMARY:\n", result["top_paper_summary"])


result["entities"]


result["keywords"]
